In [1]:
#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Set your project directory
import os

project_path = '/content/drive/MyDrive/Colab Notebooks'
os.chdir(project_path)

print(os.listdir())

['Neuron.ipynb', 'main', 'dataset', 'results', 'Text_Summarizer.ipynb']


In [3]:
!pip install transformers

In [4]:
!pip install "transformers[torch]"

In [5]:
# Import Libraries
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [6]:
# Load the data
train_data = pd.read_csv('dataset/samsum-train.csv')
val_data = pd.read_csv('dataset/samsum-validation.csv')


In [7]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [8]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [9]:
train_data.sample(5)

,id,dialogue,summary
5216,13865272,Molly: Wanna do some shopping on Saturday?\nKi...,Molly has to buy a wedding dress. She got enga...
8811,13730968-1,"Jaxon: man, that was fucked up\r\nTaylor: yea ...",Jaxon is very disappointed with the play the t...
7644,13731093,Helene: I'm outside the library\r\nBrad: great...,Brad will meet Helene at the library in 5 minu...
11086,13611995,Jane: Hello.\r\nLarry: Oh no I just need to te...,Larry won't manage to have a date with Jane to...
10583,13611658-1,James: hiya do you know whats wring with our w...,James had problems with washing machine. Sue i...


In [10]:
train_data.shape

(14732, 3)

In [11]:
val_data.shape

(818, 3)

In [12]:
# Random sampling for training data
train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)

# Random sampling for validation data
valid_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [13]:
print(f"trained data size: {train_data.shape}, valid data size: {valid_data.shape}")

trained data size: (5000, 3), valid data size: (500, 3)


In [14]:
# Data Preprocessing
import re  # Regular Expression Library

def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)  # Remove HTML tags
    text = re.sub(r"[\r\n]+", " ", text)  # Replace line breaks with spaces
    text = re.sub(r"\s+", " ", text)  # Remove extra spaces
    text = text.strip().lower()  # Remove surrounding spaces and convert to lowercase
    return text

In [15]:
# Clean Training Data
train_data['dialogue']=train_data['dialogue'].apply(clean_data)
train_data['summary']=train_data['summary'].apply(clean_data)

# Clean Validation Data
valid_data['dialogue']=valid_data['dialogue'].apply(clean_data)
valid_data['summary']=valid_data['summary'].apply(clean_data)


In [16]:
train_data['dialogue'][0]


"violet: hi! i came across this austin's article and i thought that you might find it interesting violet: claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

**Tokenization**

In [17]:
# from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
tokenizer = T5Tokenizer.from_pretrained("t5-small") # light weight version for t5 model


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [18]:
# raw data to tokenized input for model fine tuning
#  take raw data and  tokenize both dialogue and summary and combine these for model training inputs
def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding='max_length',max_length=512,truncation=True)
  targets=tokenizer(data['summary'],padding='max_length',max_length=150,truncation=True)
  # combine the target with inputs
  inputs['labels']=targets['input_ids'] # token id -> add to  input as labels
  return inputs


In [19]:
train_dataset=train_data.apply(tokenize, axis=1).tolist() # easy for huggingface
valid_dataset=valid_data.apply(tokenize, axis=1).tolist()

In [59]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [60]:
valid_dataset[0]

{'input_ids': [3, 15, 26, 26, 10, 2275, 210, 6, 410, 25, 1616, 24, 79, 31, 60, 3, 23560, 178, 12, 3, 9, 315, 3066, 58, 4659, 10, 14228, 9, 9, 9, 9, 144, 3, 10, 32, 4659, 10, 150, 55, 213, 31, 26, 25, 1616, 24, 58, 3, 15, 26, 26, 10, 168, 6, 34, 31, 7, 882, 2314, 3, 15, 26, 26, 10, 11, 13515, 131, 1219, 178, 4659, 10, 11, 103, 25, 214, 125, 34, 1112, 21, 178, 58, 3, 15, 26, 26, 10, 79, 751, 31, 17, 483, 8, 5812, 7, 3, 15, 26, 26, 10, 68, 3, 23, 214, 8, 16879, 56, 129, 7873, 972, 4659, 10, 11, 3, 23, 3382, 24, 19, 3, 9, 888, 24, 19, 5741, 12, 143, 762, 1842, 3, 15, 26, 26, 10, 17945, 6, 3382, 78, 3, 15, 26, 26, 10, 79, 43, 3, 9, 6613, 194, 13, 1705, 3, 31, 235, 143, 378, 1842, 31, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
# inputs ids -> dialogue tokenized version
  # 1 is for End of Sequence and 0 are the padding
# attention mask -> which valid value in input and padding values

# labels - target -> summary token ids

In [20]:
print(len(train_dataset[0]['input_ids']))
print(len(train_dataset[0]['labels']))


512
150


In [21]:
type(train_dataset)
type(valid_dataset)

list

In [22]:
# Start Working With Model
          # NLP TASK ALSO GENERATING NEW TEXT And dependent on inputs thats way conditional generation

model= T5ForConditionalGeneration.from_pretrained('t5-small')


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [23]:
# Fine Tune Model
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: cuda


In [24]:
# Important Arguments
training_args=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy='epoch',
    save_strategy='epoch',

    warmup_steps=500 # lr  starting mein slow and baad main faster
)


In [25]:
# build the trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset
)

In [26]:
# TRAIN THE MODEL
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.661007,0.370744
2,0.399620,0.355064
3,0.376138,0.348679
4,0.360031,0.346265
5,0.353718,0.344956
6,0.349148,0.344947


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.8053047892252604, metrics={'train_runtime': 1492.2357, 'train_samples_per_second': 20.104, 'train_steps_per_second': 2.513, 'total_flos': 4060254044160000.0, 'train_loss': 0.8053047892252604, 'epoch': 6.0})

In [27]:
# model load-> fine tune -> save the model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained('./saved_summary_model')




Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [33]:
# Load the fine-tuned model and tokenizer

trained_model = T5ForConditionalGeneration.from_pretrained(
    './saved_summary_model'
)

tokenizer = T5Tokenizer.from_pretrained(
    './saved_summary_model'
)

# Move model to the selected device
trained_model = trained_model.to(device)

# Set model to evaluation mode
trained_model.eval()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [35]:
# Test the core logic for summarization

def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    # # Add T5 task prefix
    # dialogue = "summarize: " + dialogue

    # Tokenize
    inputs = tokenizer(
        dialogue,
        padding='max_length',
        max_length=512,
        truncation=True,
        return_tensors='pt'
    ).to(device)  # Move input tensors to GPU/CPU

    # Generate summary token IDs
    with torch.no_grad():
        targets = trained_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=150,
            num_beams=4,
            early_stopping=True
        )

    # Decode token IDs into text
    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary

In [36]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. ai systems are becoming more capable due to advances in deep learning and access to large datasets. they can perform complex tasks such as language understanding, image recognition, and even code generation.
